In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 1: Install packages
print("Installing required packages...\n")
!pip install -q transformers datasets peft accelerate bitsandbytes
print("Installation complete!")

Installing required packages...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.8 MB/s eta 0:00:00
Installation complete!


In [ ]:
#updated

!pip install -qU langchain-text-splitters

In [ ]:
from huggingface_hub import login
import os

# 1. Clear any environment variable that might be stuck
if "HF_TOKEN" in os.environ:
    del os.environ["HF_TOKEN"]

# 2. Re-login (Use a token with "WRITE" permission from HF settings)
# This will prompt you to enter your token.
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import notebook_login


# This will prompt you for your token. Paste it there.
notebook_login()

In [ ]:
!cat /content/drive/MyDrive/GL_CAP_PROJECT/dataset_info.json

{
  "medical_combined_train": {
    "file_name": "../medical_datasets/medical_combined_train.json",
    "formatting": "alpaca",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "medical_combined_val": {
    "file_name": "../medical_datasets/medical_combined_validation.json",
    "formatting": "alpaca",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "medical_m2": {
    "file_name": "m2_augmented_train.json",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}

In [ ]:
# CELL 2: Import libraries
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import os

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
# CELL 3: Load your JSON files
# /content/drive/MyDrive/GL_CAP_PROJECT
print("Loading medical datasets...\n")

# CHANGE THESE PATHS if your dataset name is different
with open('/content/drive/MyDrive/GL_CAP_PROJECT/medical_combined_train.json', 'r') as f:
    train_data = json.load(f)

with open('/content/drive/MyDrive/GL_CAP_PROJECT/medical_combined_validation.json', 'r') as f:
    val_data = json.load(f)

with open('/content/drive/MyDrive/GL_CAP_PROJECT/medical_combined_test.json', 'r') as f:
    test_data = json.load(f)

print(f"Train: {len(train_data)} samples")
print(f"Validation: {len(val_data)} samples")
print(f"Test: {len(test_data)} samples")
print(f"Total: {len(train_data) + len(val_data) + len(test_data)} samples")

Loading medical datasets...

Train: 359 samples
Validation: 45 samples
Test: 45 samples
Total: 449 samples


What is LoRA Training?

LoRA stands for Low-Rank Adaptation.

It is a parameter-efficient fine-tuning (PEFT) method used to adapt large language models (LLMs) like LLaMA, GPT, etc., without updating all model weights.

Instead of retraining billions of parameters, LoRA:

- Freezes the original model weights

- Adds small trainable low-rank matrices

- Trains only those small matrices

#### Normal Fine-Tuning:

Input → [W] → Output
        (update all W)



#### LoRA Fine-Tuning:

Input → [W (frozen)]
          +
        [A × B] (trainable small matrices)
          ↓
        Output
W is frozen. Only A and B are trained.

What Are Adapter Files?

When you train a model using LoRA, you do NOT modify the full base model weights.

Instead:

- Base model → stays frozen

- Only small LoRA layers are trained

- The trained LoRA weights are saved separately

Those saved small weights are called:

Adapter files

Adapter files store:

The learned low-rank matrices (A and B)

In [ ]:
train_data[:1]

[{'instruction': 'Based on the following medical context, answer this question:\n\nQuestion: Do follow-up recommendations for abnormal Papanicolaou smears influence patient adherence?\n\nContext: To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results. Retrospective cohort study. Three northern California family planning clinics. All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears. Routine clinic protocols for follow-up--telephone call, letter, or certified letter--were applied without regard to the type of abnormality seen on a Pap smear or recommended examination. Documented adherence to follow-up within 8 months of an abnormal result. Attempts to contact the patien

In [ ]:
# CELL 4: Load model and tokenizer FIRST
# Loads a 3B LLaMA model in 4-bit compressed form and prepares it for LoRA-based fine-tuning using QLoRA.
print("Loading Llama-3.2-3B-Instruct model and tokenizer...\n")

model_name = "meta-llama/Llama-3.2-3B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

print("Model and tokenizer loaded!")

Loading Llama-3.2-3B-Instruct model and tokenizer...



config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model and tokenizer loaded!


In [ ]:
# CELL 5: Apply LoRA
# This code:
# Creates a LoRA configuration
# Injects LoRA adapters into specific transformer layers
# Freezes original weights
# Makes only small LoRA matrices trainable
# So instead of training billions of parameters…
# You train only tiny adapter matrices.
print("Configuring LoRA...\n")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA applied!")

Configuring LoRA...

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

LoRA applied!


In [ ]:
# CELL 6: Format and tokenize data (CRITICAL - THIS IS WHERE PADDING HAPPENS)
# cell is doing prompt construction + tokenization + fixed padding + label creation for your train/val/test datasets.
print("Formatting and tokenizing data...\n")

def format_and_tokenize(sample):
    """Format sample into prompt and tokenize with FIXED padding."""
    instruction = sample['instruction']
    input_text = sample.get('input', '')
    output = sample['output']

    # Create prompt
    if input_text:
        prompt = f"""<|system|>
You are a helpful medical assistant that provides accurate and concise answers to medical questions.</s>
<|user|>
{instruction}
{input_text}</s>
<|assistant|>
{output}</s>"""
    else:
        prompt = f"""<|system|>
You are a helpful medical assistant that provides accurate and concise answers to medical questions.</s>
<|user|>
{instruction}</s>
<|assistant|>
{output}</s>"""

    # Tokenize with FIXED padding to max_length
    # This ensures ALL sequences are exactly 512 tokens
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding="max_length",  # KEY: Pad to max_length NOW
        return_tensors=None
    )

    # Set labels (same as input_ids)
    result["labels"] = result["input_ids"].copy()

    return result

# Convert to HF Dataset and apply formatting
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

# Apply tokenization (this removes original columns)
train_dataset = train_dataset.map(
    format_and_tokenize,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

val_dataset = val_dataset.map(
    format_and_tokenize,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

test_dataset = test_dataset.map(
    format_and_tokenize,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test"
)

print("\nTokenization complete!")
print(f"Columns: {train_dataset.column_names}")
print(f"Train samples: {len(train_dataset)}")
print(f"All sequences are exactly {len(train_dataset[0]['input_ids'])} tokens")

Formatting and tokenizing data...



Tokenizing train:   0%|          | 0/359 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/45 [00:00<?, ? examples/s]


Tokenization complete!
Columns: ['input_ids', 'attention_mask', 'labels']
Train samples: 359
All sequences are exactly 512 tokens


In [ ]:
# CELL 7: Training configuration
print("Setting up training...\n")

#output_dir = "./medical_qa_model"
output_dir = "/content/drive/MyDrive/GL_CAP_PROJECT/medical_qa_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to="none"
)

print("Training config ready!")
print(f"Will train for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size}")

Setting up training...

Training config ready!
Will train for 3 epochs
Batch size: 4


In [ ]:
# CELL 8: Create simple data collator
# Since all sequences are already padded to 512, we use default collator
from transformers import default_data_collator

print("Using default_data_collator (sequences already padded)")

Using default_data_collator (sequences already padded)


In [ ]:
# CELL 9: Initialize trainer
print("Initializing trainer...\n")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator
)

print("Trainer ready!")
print("\n" + "="*70)
print("READY TO TRAIN")
print("="*70)

Initializing trainer...

Trainer ready!

READY TO TRAIN


In [ ]:
# CELL 10: TRAIN!
print("\nStarting training...\n")

print("="*70)

trainer.train()

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)


Starting training...



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.381897,1.378188
2,1.318969,1.348133
3,1.118168,1.362652


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



TRAINING COMPLETE!


In [ ]:
# CELL 11: Save model
print("Saving model...\n")

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")
print("\nSaved files:")
for file in os.listdir(output_dir):
    if os.path.isfile(os.path.join(output_dir, file)):
        print(f"   {file}")

Saving model...

Model saved to /content/drive/MyDrive/GL_CAP_PROJECT/medical_qa_model

Saved files:
   README.md
   adapter_config.json
   chat_template.jinja
   tokenizer_config.json
   tokenizer.json
   adapter_model.safetensors


In [ ]:
# CELL 12: Test the model
print("="*70)
print("TESTING MODEL")
print("="*70)
print()

test_queries = [
    "What are the common symptoms of type 2 diabetes?",
    "Explain what hypertension is.",
    "What causes dehydration?",
    "How eyes gets impacted due to Diabetes?"
]

for i, query in enumerate(test_queries, 1):
    print(f"[{i}/{len(test_queries)}] Query: {query}")
    print("-"*70)

    prompt = f"""<|system|>
You are a helpful medical assistant that provides accurate and concise answers to medical questions.</s>
<|user|>
{query}</s>
<|assistant|>
"""
    # Text -> Token IDs -> Sends to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # This is the actual inference step
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # This converts: Token IDs -> Text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # only want the assistant answer.
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()

    print(f"Response: {response}\n")
    print("="*70)
    print()

print("Testing complete!")

TESTING MODEL

[1/4] Query: What are the common symptoms of type 2 diabetes?
----------------------------------------------------------------------
Response: Common symptoms of type 2 diabetes are:

1. Increased thirst and urination
2. Blurred vision
3. Fatigue
4. Recurring infections
5. Skin problems
6. Fluctuations in blood sugar levels

Type 2 diabetes is a chronic condition that affects many people, but many of them do not have any symptoms at all. It is a condition that develops over time, and if you have risk factors, you should get tested for it even if you do not have any symptoms.</s>


[2/4] Query: Explain what hypertension is.
----------------------------------------------------------------------
Response: Hypertension is high blood pressure. It is the most common cause of stroke and heart attack and is a risk factor for heart failure and kidney disease.</s>


[3/4] Query: What causes dehydration?
----------------------------------------------------------------------
Respons

In [ ]:
# ========================================
# MODEL EVALUATION
# ========================================


import json
import torch
from tqdm.auto import tqdm
import numpy as np
from collections import defaultdict
import random

print("="*70)
print("MODEL EVALUATION")
print("="*70)
print()

# ========================================
# 1. QUICK PERFORMANCE CHECK
# ========================================
print(" Quick Performance Check")
print("-"*70)

# Evaluate on validation set using trainer
print("\nCalculating validation metrics...")
eval_results = trainer.evaluate()

print("\n Validation Metrics:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

print("\n" + "="*70)

# ========================================
# 2. TEST SET PREDICTIONS
# ========================================
print("\n Test Set Evaluation")
print("-"*70)
print(f"\nGenerating predictions for {len(test_data)} test samples...")

model.eval()
predictions = []
correct = 0
total_scored = 0

for idx in tqdm(range(min(len(test_data), 45)), desc="Testing"):  # Test on all 45
    sample = test_data[idx]
    instruction = sample['instruction']
    expected = sample['output']

    # Create prompt
    prompt = f"""<|system|>
You are a helpful medical assistant that provides accurate and concise answers to medical questions.</s>
<|user|>
{instruction}</s>
<|assistant|>
"""

    # Generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in generated:
        generated = generated.split("<|assistant|>")[-1].strip()

    predictions.append({
        'question': instruction[:100] + "...",
        'expected': expected,
        'generated': generated
    })

    # Simple accuracy for yes/no/maybe questions
    if expected.lower().startswith(('yes', 'no', 'maybe')):
        expected_ans = expected.lower().split('.')[0].strip()
        if expected_ans in generated.lower()[:30]:
            correct += 1
        total_scored += 1

print("\n Predictions complete!")

# ========================================
# 3. RESULTS SUMMARY
# ========================================
print("\n" + "="*70)
print(" RESULTS SUMMARY")
print("="*70)

if total_scored > 0:
    accuracy = (correct / total_scored) * 100
    print(f"\n Accuracy (Yes/No/Maybe questions): {accuracy:.1f}% ({correct}/{total_scored})")
else:
    print("\n Test set contains mostly open-ended questions")

# Response lengths
gen_lengths = [len(p['generated'].split()) for p in predictions]
exp_lengths = [len(p['expected'].split()) for p in predictions]

print(f"\n📏 Response Lengths:")
print(f"   Model avg: {np.mean(gen_lengths):.1f} words")
print(f"   Expected avg: {np.mean(exp_lengths):.1f} words")
print(f"   Ratio: {np.mean(gen_lengths)/np.mean(exp_lengths):.2f}x")

# Check completeness
complete = sum(1 for p in predictions if len(p['generated']) > 20)
print(f"\n Complete responses: {complete}/{len(predictions)} ({complete/len(predictions)*100:.1f}%)")

print("\n" + "="*70)

# ========================================
# 4. SAMPLE OUTPUTS
# ========================================
print("\n Sample Predictions (5 random examples)")
print("="*70)

for i, idx in enumerate(random.sample(range(len(predictions)), min(5, len(predictions))), 1):
    pred = predictions[idx]
    print(f"\n--- EXAMPLE {i} ---")
    print(f"\n Question: {pred['question']}")
    print(f"\n Expected: {pred['expected'][:150]}...")
    print(f"\n Generated: {pred['generated'][:150]}...")
    print()

print("="*70)

# ========================================
# 5. SAVE RESULTS
# ========================================
print("\n Saving evaluation results...")

# Save predictions
with open('/content/drive/MyDrive/GL_CAP_PROJECT/test_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

# Save summary
summary = {
    'validation_loss': eval_results.get('eval_loss', 'N/A'),
    'test_samples': len(predictions),
    'accuracy': f"{accuracy:.1f}%" if total_scored > 0 else 'N/A',
    'avg_response_length': f"{np.mean(gen_lengths):.1f} words",
    'complete_responses': f"{complete}/{len(predictions)}"
}

with open('/content/drive/MyDrive/GL_CAP_PROJECT/evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(" Results saved!")
print("test_predictions.json - All predictions")
print("evaluation_summary.json - Summary metrics")

print("\n" + "="*70)
print(" EVALUATION COMPLETE!")
print("="*70)

# Display summary
print("\n FINAL SUMMARY:")
print(f"   Training completed")
print(f"   Validation loss: {eval_results.get('eval_loss', 'N/A'):.4f}" if 'eval_loss' in eval_results else "   Validation loss: N/A")
print(f"   Test accuracy: {summary['accuracy']}" if total_scored > 0 else "   Test accuracy: N/A (open-ended questions)")
print(f"   Avg response quality: {'Good' if complete/len(predictions) > 0.8 else 'Fair'}")
print()

MODEL EVALUATION

 Quick Performance Check
----------------------------------------------------------------------

Calculating validation metrics...



 Validation Metrics:
   eval_loss: 1.3481
   eval_runtime: 12.8436
   eval_samples_per_second: 3.5040
   eval_steps_per_second: 0.9340
   epoch: 3.0000


 Test Set Evaluation
----------------------------------------------------------------------

Generating predictions for 45 test samples...


Testing:   0%|          | 0/45 [00:00<?, ?it/s]


 Predictions complete!

 RESULTS SUMMARY

 Accuracy (Yes/No/Maybe questions): 47.6% (10/21)

📏 Response Lengths:
   Model avg: 167.6 words
   Expected avg: 71.0 words
   Ratio: 2.36x

 Complete responses: 45/45 (100.0%)


 Sample Predictions (5 random examples)

--- EXAMPLE 1 ---

 Question: Test for diagnosis of pyogenic meningitis is?

Options:
A. Widal
B. CSF PCR
C. CSF examination
D. PE...

 Expected: The correct answer is B: CSF PCR

Explanation: ANSWER: (C) CSF examinationREF: Harrison 17th ed chapter 376The diagnosis of bacterial meningitis is ma...

 Generated: The correct answer is C: CSF examination

Explanation: CSF PCR is a rapid and sensitive method for detecting the presence of pyogenic meningitis. CSF ...


--- EXAMPLE 2 ---

 Question: Summarize the key information from the following article, focusing on any scientific or health-relat...

 Expected: Re-designed Fuji Speedway to host its first Japanese Grand Prix since 1977 .
Suzuka to alternate with Fuji from 2009 .
Co